# 02 · The lens: when does the model know the answer?

A transformer's residual stream is the same space at every depth. So we can take the stream **after any block** and push it through the final LayerNorm and output head, as if the model stopped there (the "logit lens" of language models).

If the decoded tendency already looks like the final one at block 3, the later blocks are refining it. If it only appears at block 6, that is where the work is done.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
x = pg.state("2010-01-15T00")
decoded, table = pg.lens(em, x)
table.round(3)          # RMSE vs the final output, relative to its RMS (0 = already the answer)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for g in ["PS/TS", "T", "U", "V", "water"]:
    ax.plot(table["after"], table[g], marker="o", label=g)
ax.set_ylabel("relative error vs final output"); ax.set_xticks(range(len(table))); ax.set_xticklabels(table["after"], rotation=45); ax.legend();

What the V_3 tendency looks like as it is built up, block by block:

In [ ]:
c = pg.ch("V_3")
show = [3, 5, 6, 8]          # index 0 = embedding only, k = after block k-1
pg.plot_maps({table["after"][k]: decoded[k][0, c] for k in show}, shared=True, cols=2);

## Try this

1. Which field group is "decided" earliest? Is moisture harder (later) than temperature?
2. Run the lens on a **summer** date and a **tropical cyclone** case (look for strong V_3 dipoles in the tropics). Does the depth at which the answer appears change?
3. The lens uses the final LayerNorm on an intermediate stream, which may be unfair. A "tuned lens" fits one linear map per layer. Fit one by least squares on a few dates: does block 2 then decode much better?
4. Compare the lens with ablation (notebook 03): a block whose removal hurts most should be where the lens error drops most. Does that hold?